In [1]:
print("Hello, world!")

Hello, world!


In [5]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

In [6]:


# 1. Load real-world Kaggle PJM Hourly Energy Consumption data
DATA_URL = "https://raw.githubusercontent.com/archd3sai/Hourly-Energy-Consumption-Prediction/master/PJME_hourly.csv"
df = pd.read_csv(DATA_URL, parse_dates=['Datetime'], index_col='Datetime')
df = df.sort_index() # Ensure exact chronological order


In [7]:
df.head()

,PJME_MW
Datetime,
2002-01-01 01:00:00,30393.0
2002-01-01 02:00:00,29265.0
2002-01-01 03:00:00,28357.0
2002-01-01 04:00:00,27899.0
2002-01-01 05:00:00,28057.0


In [8]:

# 2. Extract and normalize the target consumption column
raw_values = df['PJME_MW'].values.astype(np.float32)
min_val, max_val = raw_values.min(), raw_values.max()
normalized_values = (raw_values - min_val) / (max_val - min_val)

In [9]:


# 3. Construct historical sequence windows (24-hour lookback window to predict hour 25)
def create_rolling_windows(data, seq_length):
    xs, ys = [], []
    for i in range(len(data) - seq_length):
        xs.append(data[i:(i + seq_length)])
        ys.append(data[i + seq_length])
    return np.array(xs), np.array(ys)

SEQ_LENGTH = 24
X, y = create_rolling_windows(normalized_values, SEQ_LENGTH)

In [11]:


# 4. Build PyTorch structural abstractions
class ElectricityDataset(Dataset):
    def __init__(self, sequences, targets):
        self.sequences = torch.tensor(sequences, dtype=torch.float32).unsqueeze(-1)
        self.targets = torch.tensor(targets, dtype=torch.float32).unsqueeze(-1)
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return self.sequences[idx], self.targets[idx]

# Subsample data to accelerate training execution loop
dataset = ElectricityDataset(X[:2000], y[:2000])
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

print(f"Loaded PJME Dataset. Feature shape: {X.shape}, Target shape: {y.shape}")

Loaded PJME Dataset. Feature shape: (145342, 24), Target shape: (145342,)


In [16]:
import torch.nn as nn

class ElectricityForecaster(nn.Module):
    def __init__(self, cell_type, input_size, hidden_size, output_size):
        super().__init__()
        self.cell_type = cell_type
        
        # Initialize selected recurrent cell using exact parameters
        if cell_type == 'RNN':
            self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        elif cell_type == 'LSTM':
            self.rnn = nn.LSTM(input_size, hidden_size, batch_first=True)
        elif cell_type == 'GRU':
            self.rnn = nn.GRU(input_size, hidden_size, batch_first=True)
        else:
            raise ValueError("Select 'RNN', 'LSTM', or 'GRU'")
            
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        # x shape: (batch_size, seq_length, input_size)
        out, _ = self.rnn(x)
        
        # Decode the final step's hidden vector position mapping
        out = self.fc(out[:, -1, :])
        return out

In [17]:
def train_model(model, dataloader, epochs=3, lr=0.001):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    print(f"--- Training {model.cell_type} Framework ---")
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for seqs, targets in dataloader:
            optimizer.zero_grad()
            outputs = model(seqs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        print(f"Epoch {epoch+1}/{epochs} | Mean Loss: {total_loss/len(dataloader):.5f}")

# Train and validate specific structural configurations
lstm_net = ElectricityForecaster(cell_type='LSTM', input_size=1, hidden_size=32, output_size=1)
train_model(lstm_net, dataloader)

gru_net = ElectricityForecaster(cell_type='GRU', input_size=1, hidden_size=32, output_size=1)
train_model(gru_net, dataloader)

--- Training LSTM Framework ---
Epoch 1/3 | Mean Loss: 0.01409
Epoch 2/3 | Mean Loss: 0.00619
Epoch 3/3 | Mean Loss: 0.00565
--- Training GRU Framework ---
Epoch 1/3 | Mean Loss: 0.01889
Epoch 2/3 | Mean Loss: 0.00630
Epoch 3/3 | Mean Loss: 0.00546


In [19]:
def execute_training(model, loader, epochs=3, lr=0.001):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    print(f"\n--- Training {model.cell_type} Architecture ---")
    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        for seqs, targets in loader:
            optimizer.zero_grad()
            
            outputs = model(seqs)
            loss = criterion(outputs, targets)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(loader):.5f}")

# Instantiate and train models by explicitly passing initial arguments
lstm_network = ElectricityForecaster(cell_type='LSTM', input_size=1, hidden_size=32, output_size=1)
execute_training(lstm_network, dataloader)

# Instantiate and train GRU variants
gru_network = ElectricityForecaster(cell_type='GRU', input_size=1, hidden_size=32, output_size=1)
execute_training(gru_network, dataloader)


--- Training LSTM Architecture ---
Epoch 1/3 | Loss: 0.08227
Epoch 2/3 | Loss: 0.00691
Epoch 3/3 | Loss: 0.00659

--- Training GRU Architecture ---
Epoch 1/3 | Loss: 0.05931
Epoch 2/3 | Loss: 0.00488
Epoch 3/3 | Loss: 0.00441
